In [ ]:
!pip install bitsandbytes trl

In [ ]:
import os
import torch
from contextlib import nullcontext
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
from trl import SFTConfig,SFTTrainer
from datasets import load_dataset

양자화된 베이스 모델 로드하기

In [ ]:
#양자화된 모델 로드
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32
)
repo_id='microsoft/Phi-3-mini-4k-instruct'
model=AutoModelForCausalLM.from_pretrained(
    repo_id,device_map="cuda:0",quantization_config=bnb_config
)

In [ ]:
#model이 차지하는 메모리(MB단위)
print(model.get_memory_footprint()/1e6)

2206.341504


양자화 과정은 주로 transformer decoder block에 있는 linear layer을 대상으로 한다.

In [ ]:
#model의 구조
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear4bit(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear4bit(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=

Linear4bit layer는 메모리를 적게 차지하지만 업데이트할 수 없다. 그렇기에 어댑터를 추가하여 모델의 동작을 바꾼다.

LoRA설정

In [ ]:
#양자화된 layer 각각에 LoRA 어댑터 추가

#훈련 과정에서 수치 안정성을 향상시킴
model = prepare_model_for_kbit_training(model)
config=LoraConfig(
    #어댑터의 rank가 작을수록 훈련할 parameter가 적다.
    r=8,
    lora_alpha=16, #일반적으로 2*r
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    #대상 모듈 지정
    target_modules=["o_proj","qkv_proj","gate_up_proj","down_proj"],
)

model=get_peft_model(model,config)
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

In [ ]:
#양자화되지 않은 다른 모든 layer는 FP32로 표현되므로 모델의 메모리 사용량이 20%정도 증가한다.
print(model.get_memory_footprint()/1e6)

2651.074944


In [ ]:
train_p, tot_p = model.get_nb_trainable_parameters()
print(f"훈련 가능한 파라미터:           {train_p/1e6:.2f}M")
print(f"총 파라미터:                  {tot_p/1e6:.2f}M")
print(f"훈련 가능한 파라미터의 비율(%):   {100*train_p/tot_p:.2f}%")

훈련 가능한 파라미터:           12.58M
총 파라미터:                  3833.66M
훈련 가능한 파라미터의 비율(%):   0.33%


data formating

In [ ]:
#yoda체 문장 로드
dataset=load_dataset("dvgodoy/yoda_sentences",split="train")
dataset

Dataset({
    features: ['sentence', 'translation', 'translation_extra'],
    num_rows: 720
})

In [ ]:
dataset[0]

{'sentence': 'The birch canoe slid on the smooth planks.',
 'translation': 'On the smooth planks, the birch canoe slid.',
 'translation_extra': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}

In [ ]:
#dataset을 대화 포맷으로 변경한다.
def format_dataset(examples):
  if isinstance(examples["prompt"],list):
    output_texts=[]
    for i in range(len(examples["prompt"])):
      converted_sample = [
          {"role": "user", "content": examples["prompt"][i]},
          {"role":"assistant","content":examples["completion"[i]]},
      ]
      output_texts.append(converted_sample)
    return {"messages":output_texts}
  else:
    converted_sample=[
        {"role":"user","content":examples["prompt"]},
        {"role":"assistant","content":examples["completion"]},
    ]
    return {"messages":converted_sample}

In [ ]:
dataset=dataset.rename_column("sentence","prompt")
dataset=dataset.rename_column("translation_extra","completion")
dataset=dataset.map(format_dataset)
dataset=dataset.remove_columns(["prompt","completion","translation"])
messages=dataset[0]["messages"]
messages

[{'role': 'user', 'content': 'The birch canoe slid on the smooth planks.'},
 {'role': 'assistant',
  'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}]

tokenizer로드

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(repo_id)
#Phi-3 모델은 EOS token이 PAD token으로도 쓰이는 데 이를 막기 위해
#UNK token을 PAD token으로 할당한다.
tokenizer.pad_token=tokenizer.unk_token
tokenizer.pad_token_id=tokenizer.unk_token_id

tokenizer.chat_template

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

"{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'user' %}{{'<|user|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>\n' + message['content'] + '<|end|>\n'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% else %}{{ eos_token }}{% endif %}"

In [ ]:
print(tokenizer.apply_chat_template(messages,tokenize=False))

#각 대화는 <|user|> 또는 <|assistant|>으로 시작하고 <|end|>로 끝난다.
#<|endoftext|>는 전체 block의 끝을 나타낸다.

<|user|>
The birch canoe slid on the smooth planks.<|end|>
<|assistant|>
On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>
<|endoftext|>


SFTTrainer를 사용하여 Fine-Tuning

모델의 규모에 상광없이 Fine-Tuning은 모델을 처음부터 훈련하는 것과 정확히 동일한 훈련 과정을 거친다.

In [ ]:
#SFTTrainer의 매개변수는 다음과 같다.
#모델, tokenizer, dataset, config

#SFTConfig
sft_config=SFTConfig(
    #메모리 절약
    gradient_checkpointing=True,
    #파이토치 새 버전에서 예외를 피하기 위해 지정
    gradient_checkpointing_kwargs={"use_reentrant":False},
    #gradient 누적과 배치 크기
    #(업데이트를 위한) 실제 배치 크기는 마이크로 배치 크기와 같다.
    gradient_accumulation_steps=1,
    #초기 (마이크로) 배치 크기
    per_device_train_batch_size=16,
    #배치 크기가 메모리 부족을 일으키면 문제가 해결될 때까지 반으로 나눈다.
    auto_find_batch_size=True,

    #dataset 설정
    max_length=64,
    #dataset packing을 한다는 것은 padding이 필요 없다는 의미이다.
    packing=True,
    packing_strategy="wrapped",

    #일반적인 훈련 매개변수
    num_train_epochs=10,
    learning_rate=3e-4,
    #8-bit Adam optimizer(LoRA를 사용하는 경우 큰 도움이 되지 않다.)
    optim="paged_adamw_8bit",

    #logging parameter
    logging_steps=10,
    logging_dir="./logs",
    output_dir="./phi3-mini-yoda-adapter",
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
#trainer object 생성
trainer=SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=dataset,
)

Tokenizing train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

In [ ]:
dl=trainer.get_train_dataloader()
batch=next(iter(dl))

In [ ]:
batch["input_ids"][0],batch["labels"][0]

(tensor([ 3974, 29892,  4337,   278,   325,   271, 29892,   366,  1818, 29889,
         32007, 32000, 32010,   450,   289,   935,   310,   278,   282,   457,
          5447,   471,   528,  4901,   322,  6501, 29889, 32007, 32001, 26399,
          1758,  4317, 29889,  1383,  4901,   322,  6501, 29892,   278,   289,
           935,   310,   278,   282,   457,  5447,   471, 29889, 32007, 32000,
         32010,   951,  5989,  2507, 17354,   322, 13328,   297,   278,  6416,
         29889, 32007, 32001,   512], device='cuda:0'),
 tensor([ 3974, 29892,  4337,   278,   325,   271, 29892,   366,  1818, 29889,
         32007, 32000, 32010,   450,   289,   935,   310,   278,   282,   457,
          5447,   471,   528,  4901,   322,  6501, 29889, 32007, 32001, 26399,
          1758,  4317, 29889,  1383,  4901,   322,  6501, 29892,   278,   289,
           935,   310,   278,   282,   457,  5447,   471, 29889, 32007, 32000,
         32010,   951,  5989,  2507, 17354,   322, 13328,   297,   278,  64

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,2.849109
20,1.829733
30,1.605833
40,1.522166
50,1.399264
60,1.295499
70,1.191498
80,0.992769
90,0.896720
100,0.631070


Step,Training Loss
10,2.849109
20,1.829733
30,1.605833
40,1.522166
50,1.399264
60,1.295499
70,1.191498
80,0.992769
90,0.896720
100,0.631070


TrainOutput(global_step=220, training_loss=0.8370171080936085, metrics={'train_runtime': 2811.3209, 'train_samples_per_second': 1.213, 'train_steps_per_second': 0.078, 'total_flos': 4890970340720640.0, 'train_loss': 0.8370171080936085})

model에 질의하기

In [ ]:
#prompt formatting
#마지막에 <|assistant|> 추가
def gen_prompt(tokenizer,sentence):
  converted_sample=[{"role":"user","content":sentence}]
  prompt=tokenizer.apply_chat_template(
      converted_sample,tokenize=False,add_generation_prompt=True
  )
  return prompt

In [ ]:
#sample 문장으로 prompt를 생성하겠다.
sentence="The Force is strong in you!"
prompt=gen_prompt(tokenizer,sentence)
print(prompt)

<|user|>
The Force is strong in you!<|end|>
<|assistant|>



In [ ]:
#prompt를 tokenization하여 tokenID의 tensor를 만든다.
def generate(model, tokenizer,prompt,max_new_tokens=64,
             skip_special_tokens=False):
  tokenized_input=tokenizer(
      prompt,add_special_tokens=False,return_tensors="pt"
  ).to(model.device)

  model.eval()
  #혼합 정밀도를 사용해 훈련하는 경우 autocast context를 사용한다.
  ctx=torch.autocast(device_type=model.device.type,dtype=model.dtype) \
    if model.dtype in [torch.float16, torch.bfloat16] else nullcontext()

  with ctx:
    generation_output=model.generate(**tokenized_input,
                                     eos_token_id=tokenizer.eos_token_id,
                                     max_new_tokens=max_new_tokens)

  output=tokenizer.batch_decode(generation_output,
                                skip_special_tokens=skip_special_tokens)
  return output[0]

In [ ]:
#요다체 문장을 생성하는지 테스트
print(generate(model,tokenizer,prompt))

<|user|> The Force is strong in you!<|end|><|assistant|> Strong in you, the Force is! Yes, hrrrm.<|end|><|endoftext|>


In [ ]:
#어댑터 저장
trainer.save_model("local-phi3-mini-kyoda-adapter")